# Clean (overlap-filtered) OS-data perplexity

BabyLM's `open_subtitles.train.txt` (English training data) overlaps with OPUS-100 en-hi's English side by ~25%, and en-te's English side by ~63% -- meaning the "OS-data" perplexity numbers already in `evaluation.md`/`paper.tex` are partly measuring memorized text, not genuine held-out generalization.

This notebook filters out every OPUS-100 English line that exactly matches a BabyLM training line, then recomputes perplexity on the remaining, genuinely-unseen subset -- for GPT-Wee, GPT-BERT, Llama 3.2 1B, and Sarvam-2B (mono-English and, where applicable, bilingual English).

No GPU strictly required for the filtering step; a GPU is needed for the perplexity computation itself.

In [ ]:
# Cell 1: clone the repo and install dependencies
import os

if not os.path.isdir("/content/BabyLM"):
    !git clone https://github.com/vishnup22/BabyLM.git /content/BabyLM

%cd /content/BabyLM
!git checkout evaluation
!git pull
!pip install -q torch transformers accelerate datasets huggingface_hub requests

In [ ]:
# Cell 2: log in to Hugging Face (needed for gated meta-llama/Llama-3.2-1B and private repos)
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!hf auth whoami

In [ ]:
# Cell 3: build the overlap set from BabyLM's OpenSubtitles training data, then filter
# OPUS-100 en-hi's English side down to the genuinely non-overlapping subset (this is the
# same "en" OS-data source used for every English OS-data perplexity number so far, per
# OS_OPUS_CONFIG in eval_gptbert_all.py)
import re
import requests
from datasets import load_dataset

BABYLM_OS_URL = "https://huggingface.co/datasets/BabyLM-community/BabyLM-2026-Strict/resolve/main/open_subtitles.train.txt"

def normalize(line):
    line = line.strip().lower()
    return re.sub(r"\s+", " ", line)

resp = requests.get(BABYLM_OS_URL, timeout=300)
resp.raise_for_status()
babylm_set = {normalize(l) for l in resp.text.splitlines() if normalize(l)}
print(f"BabyLM open_subtitles: {len(babylm_set):,} unique normalized lines")

ds = load_dataset("Helsinki-NLP/opus-100", "en-hi", split="train", streaming=True)
clean_texts = []
total = 0
for row in ds:
    en_text = row["translation"]["en"].strip()
    if not en_text:
        continue
    total += 1
    if normalize(en_text) not in babylm_set:
        clean_texts.append(en_text)

print(f"Full: {total:,} lines | Clean (overlap removed): {len(clean_texts):,} lines "
      f"({100*len(clean_texts)/total:.1f}% retained)")

In [ ]:
# Cell 4: shared causal perplexity function for standard HF causal LMs (GPT-2, Llama,
# Sarvam) -- same method as multilingual eval/eng_hin.py and eval_llama_missing.py:
# micro-averaged (sum NLL over all tokens, divide once, exponentiate once), max_seq_len=128,
# batch_size=32, matching the methodology already used for every other perplexity number
# in this project.
import math
import torch
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MAX_SEQ_LEN = 128
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def compute_causal_perplexity_hf(model, tokenizer, texts, max_seq_len=MAX_SEQ_LEN, batch_size=BATCH_SIZE):
    total_nll, total_tokens = 0.0, 0
    for i in tqdm(range(0, len(texts), batch_size), desc="  perplexity"):
        batch = texts[i:i + batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=max_seq_len)
        input_ids = enc["input_ids"].to(DEVICE)
        attn_mask = enc["attention_mask"].to(DEVICE)
        with torch.no_grad():
            logits = model(input_ids, attention_mask=attn_mask).logits
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = input_ids[:, 1:].contiguous()
        shift_mask = attn_mask[:, 1:].contiguous().float()
        log_probs = F.log_softmax(shift_logits, dim=-1)
        token_ll = log_probs.gather(-1, shift_labels.unsqueeze(-1)).squeeze(-1)
        total_nll += -(token_ll * shift_mask).sum().item()
        total_tokens += shift_mask.sum().item()
    return math.exp(total_nll / total_tokens) if total_tokens > 0 else float("inf")

In [ ]:
# Cell 5: clean OS-data perplexity for standard HF causal LMs -- GPT-Wee (mono-en,
# eng-hin, eng-tel), Llama 3.2 1B, Sarvam-2B
HF_MODELS = {
    "GPT-Wee mono-en":      "pulipakav-1/gpt2-english-babylm2026",
    "GPT-Wee eng-hin":      "pulipakav-1/gpt2-hin-eng_babylm2026",
    "GPT-Wee eng-tel":      "pulipakav-1/gpt2-tel-eng_babylm2026",
    "Llama 3.2 1B":         "meta-llama/Llama-3.2-1B",
    "Sarvam-2B":            "sarvamai/sarvam-2b-v0.5",
}

results = {}
for label, repo in HF_MODELS.items():
    print(f"\n=== {label} ({repo}) ===")
    tokenizer = AutoTokenizer.from_pretrained(repo)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(repo).eval().to(DEVICE)
    ppl = compute_causal_perplexity_hf(model, tokenizer, clean_texts)
    print(f"  clean OS-data perplexity: {ppl:.4f}")
    results[label] = ppl
    del model
    torch.cuda.empty_cache()

In [ ]:
# Cell 6: clean OS-data perplexity for GPT-BERT (mono-en, eng-hin, eng-tel) -- reuses
# eval_gptbert_all.py's own model-loading and causal-perplexity code directly, so the
# scoring math is guaranteed identical to every other GPT-BERT perplexity number already
# recorded in this project
%cd /content/BabyLM
import sys
sys.path.insert(0, "/content/BabyLM")
from eval_gptbert_all import load_model_and_tokenizer, compute_causal_perplexity, MODEL_REGISTRY

GPTBERT_MODELS = {
    "GPT-BERT mono-en": "mono_en",
    "GPT-BERT eng-hin":  "en_hi_seed1",
    "GPT-BERT eng-tel":  "en_tel_seed2",
}

for label, model_key in GPTBERT_MODELS.items():
    print(f"\n=== {label} ({model_key}) ===")
    spec = MODEL_REGISTRY[model_key]
    model, tokenizer, cls_id, mask_id, pad_id = load_model_and_tokenizer(spec)
    ppl = compute_causal_perplexity(model, tokenizer, clean_texts, cls_id, pad_id)
    print(f"  clean OS-data perplexity: {ppl:.4f}")
    results[label] = ppl
    del model
    torch.cuda.empty_cache()

In [ ]:
# Cell 7: summary
print(f"Clean OS-data set: {len(clean_texts):,} lines (overlap with BabyLM training removed)\n")
for label, ppl in results.items():
    print(f"  {label:20s}: {ppl:.4f}")